# Dev notebook — phase1_clean.py

Chạy từng bước của Pha 1 để xem/hiểu dữ liệu biến đổi thế nào qua mỗi hàm. Notebook này **import thẳng từ `phase1_clean.py`** (không định nghĩa lại logic) — bài học từ `iaqi_dev.ipynb`: định nghĩa lại thì dễ lệch với code thật khi sửa sau này.

Sửa `phase1_clean.py` xong, bật `%load_ext autoreload` + `%autoreload 2` bên dưới là chạy lại cell là thấy kết quả mới ngay, không cần restart kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from phase1_clean import (
    load_raw, clip_outliers, build_hourly_grid, interpolate_short_gaps,
    add_day_aggregates, print_quality_report,
    POLLUTANTS, MAX_GAP_HOURS_TO_INTERPOLATE, MIN_DAY_COMPLETENESS,
)

spark = (
    SparkSession.builder.appName("phase1_dev").master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")  # bắt buộc, xem ghi chú trong phase1_clean.py
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

INPUT = "../../data/samples/air_quality_sample.jsonl"

## Bước 1 — `load_raw()`: đọc JSONL theo schema C1 + dedup

In [ ]:
raw = load_raw(spark, INPUT)
print("so dong sau dedup:", raw.count())
raw.orderBy("station_id", "ts_epoch").show(5)

## Bước 2 — `clip_outliers()`: âm hoặc vượt trần I=500 -> null

In [ ]:
clipped = clip_outliers(raw)
for p in POLLUTANTS:
    n_null = clipped.filter(F.col(p).isNull()).count()
    print(f"{p:<6}: so gia tri bi null sau clip (am/vuot tran) = {n_null}")

Kỳ vọng: hầu hết = 0, vì sample không cố tình nhúng giá trị vượt trần (README chỉ nhúng ~5 ca âm).

## Bước 3 — `build_hourly_grid()`: lộ ra giờ thiếu HẲN

In [ ]:
grid = build_hourly_grid(clipped)
print("tong so gio sau dung luoi day du (5 tram x 2160h ky vong = 10800):", grid.count())

missing_rows = grid.filter(F.col("pm2_5").isNull()).orderBy("station_id", "ts_epoch")
print("so gio thieu (pm2_5 null):", missing_rows.count())
missing_rows.select("station_id", "ts_epoch").show(5)

## Bước 4 — `interpolate_short_gaps()`: nội suy <=3h, giữ null nếu dài hơn

In [ ]:
interpolated = interpolate_short_gaps(grid)
interpolated.groupBy("pm2_5_qc").count().show()

Xem thử 1 dòng `interpolated` cụ thể — so sánh bằng tay với giá trị trước/sau để tin tưởng công thức.

In [ ]:
one_station = interpolated.filter(F.col("station_id") == "VN_HCM_01").orderBy("ts_epoch")
example_ts = (
    one_station.filter(F.col("pm10_qc") == "interpolated")
    .select("ts_epoch").first()
)
if example_ts:
    center = example_ts["ts_epoch"]
    one_station.filter(F.col("ts_epoch").between(center - 3600 * 2, center + 3600 * 2)) \
        .select("ts_epoch", "pm10", "pm10_qc").show()
else:
    print("khong co dong interpolated nao cho pm10 o tram nay")

## Bước 5 — `add_day_aggregates()`: TB24h PM2.5/PM10 theo 'ngày AQI' (01:00 -> 00:00 hôm sau)

In [ ]:
final = add_day_aggregates(interpolated)

final.select("station_id", "ts_utc", "dt", "aqi_day").orderBy("station_id", "ts_epoch").show(3)

final.select("station_id", "aqi_day", "pm2_5_day_avg", "pm10_day_avg", "day_qc_flag") \
    .dropDuplicates(["station_id", "aqi_day"]) \
    .orderBy("station_id", "aqi_day") \
    .show(10)

Lưu ý: `aqi_day` lệch 1 giờ so với `dt` — dòng `ts_utc=...T00:00:00Z` thuộc `aqi_day` của NGÀY HÔM TRƯỚC (đúng định nghĩa QĐ 1459 mục 2.2.2a: khối 01:00 -> 00:00 hôm sau).

## Bước 6 — Báo cáo chất lượng tổng hợp

In [ ]:
print_quality_report(final)

## (Tuỳ chọn) Ghi thử ra parquet rồi đọc lại bằng pandas

In [ ]:
import shutil

OUT = "/tmp/phase1_dev_out"
shutil.rmtree(OUT, ignore_errors=True)

out_cols = (
    ["station_id", "city", "country", "lat", "lon", "ts_utc", "ts_epoch", "dt", "aqi_day"]
    + POLLUTANTS
    + ["pm2_5_day_avg", "pm10_day_avg", "day_qc_flag"]
    + [f"{p}_qc" for p in POLLUTANTS]
)
final.select(*out_cols).coalesce(1).write.mode("overwrite").partitionBy("country", "dt").parquet(OUT)

import pandas as pd
pdf = pd.read_parquet(OUT)
print(pdf.shape)
pdf.head()

## Ghi chú

- Notebook này chỉ để **đọc/hiểu/debug từng bước** — không phải nơi sửa logic. Muốn đổi hành vi thì sửa `phase1_clean.py`, cell trên tự nạp lại nhờ `%autoreload 2`.
- Nếu muốn dọn workspace sau khi test xong: `spark.stop()` và xoá `/tmp/phase1_dev_out`.